# Plot figure showing published projects

## Set up

In [ ]:
import os
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
from tableone import TableOne
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# path to datasets
base_path = os.path.join("..", "data", "physionet")

## Load the data

In [ ]:
# Custom function to parse the datetime
def parse_publish_date(date_str):
    try:
        # Try the format with microseconds first
        return datetime.strptime(date_str, "%Y-%m-%d %H:%M:%S.%f%z")
    except ValueError:
        # If that fails, try the format without microseconds
        return datetime.strptime(date_str, "%Y-%m-%d %H:%M:%S%z")

In [ ]:
projects = pd.read_csv(os.path.join(base_path, "projects.csv"), low_memory=False)

# Convert date columns to date type
# projects = projects.with_columns(pl.col("publish_date").str.to_datetime("%Y-%m-%d %H:%M:%S%.f%z", strict=False))
# projects['publish_date'] = pd.to_datetime(projects['publish_date'], errors='coerce')
# projects['publish_date'] = pd.to_datetime(projects['publish_date'], errors='coerce')
projects['publish_date'] = projects['publish_date'].apply(parse_publish_date).dt.date

# projects[projects.publish_date == pd.NaT]
projects.head()

In [ ]:
# projects['number_authors'] = projects['author_ids'].apply(safe_len)

In [ ]:
# Should we limit to latest versions only?
projects = projects[projects['is_latest_version'] == True]

In [ ]:
# Define the new projects as a dictionary
new_projects = [
    {'project_id': '0001', 'project_slug': 'mimic2db', 'publish_date': '2011-01-01', 'storage_size_mb': 717000.0},
    {'project_id': '0002', 'project_slug': 'mimic2wdb', 'publish_date': '2015-01-01', 'storage_size_mb': 6800000.0}
]

# Convert the list of dictionaries to a DataFrame
new_projects_df = pd.DataFrame(new_projects)

# Add the new projects to the existing 'projects' DataFrame
projects = pd.concat([projects, new_projects_df], ignore_index=True)

# Display the updated DataFrame
print(projects.tail())

In [ ]:
# Convert 'publish_date' to datetime if it's not already
projects['publish_date'] = pd.to_datetime(projects['publish_date'], errors='coerce')

# Extract the year from 'publish_date' and create a new column 'year'
projects['year'] = projects['publish_date'].dt.year

# Group by 'year' and count the number of projects per year
projects_per_year = projects.groupby('year').agg(count=('project_id', 'size'),
                                                 total_storage_mb=('storage_size_mb', 'sum')
                                                 ).reset_index()

projects_per_year.head(26)

## Plot functions

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def plot_projects_volume_modern(projects_per_year,
                                size_title=34,
                                size_axes_lables=34,
                                size_tick_lables=26,
                                incomplete=True,
                                bold=False):

    # Font + colours
    font_type = 'Arial Black' if bold else 'Arial'
    accent = "#4c72b0"
    accent_faded = "rgba(76,114,176,0.35)"
    text_col = "#222"
    tick_col = "#555"

    # Create the subplot figure with 1 row and 2 columns
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            "New Published Projects per Year",
            "PhysioNet Project Volume per Year"
        ),
        horizontal_spacing=0.12
    )

    # ------------------------------
    # Helper: add star to final year
    # ------------------------------
    def add_star(fig, years, values, row, col, star_size=18):
        final_year = years.iloc[-1]
        final_val = values.iloc[-1]
        fig.add_trace(
            go.Scatter(
                x=[final_year],
                y=[final_val * 1.05],
                mode="text",
                text=["★"],
                textfont=dict(size=star_size, color="#c0392b"),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=row, col=col
        )

    # Padding factor for y-axis
    padding_factor = 1.15

    years = projects_per_year["year"]
    counts = projects_per_year["count"].astype(float)
    volume_x1e3mb = (projects_per_year["total_storage_mb"] / 1000.0).astype(float)

    max_y_projects = counts.max() * padding_factor
    max_y_volume = volume_x1e3mb.max() * padding_factor

    # ------------------------------
    # Plot 1: New Published Projects
    # ------------------------------
    colors_proj = [
        accent_faded if (incomplete and i == len(counts) - 1) else accent
        for i in range(len(counts))
    ]

    fig.add_trace(
        go.Bar(
            x=years,
            y=counts,
            name="Published Projects",
            marker=dict(
                color=colors_proj,
                line=dict(color=accent, width=0.8),
            ),
            width=0.55,
        ),
        row=1, col=1
    )

    if incomplete:
        add_star(fig, years, counts, row=1, col=1)

    # ------------------------------
    # Plot 2: Project Volume
    # ------------------------------
    colors_vol = [
        accent_faded if (incomplete and i == len(volume_x1e3mb) - 1) else accent
        for i in range(len(volume_x1e3mb))
    ]

    fig.add_trace(
        go.Bar(
            x=years,
            y=volume_x1e3mb,
            name="Volume",
            marker=dict(
                color=colors_vol,
                line=dict(color=accent, width=0.8),
            ),
            width=0.55,
        ),
        row=1, col=2
    )

    if incomplete:
        add_star(fig, years, volume_x1e3mb, row=1, col=2)

    # ------------------------------
    # Overall layout
    # ------------------------------
    fig.update_layout(
        template="simple_white",
        plot_bgcolor="rgba(0,0,0,0)",
        showlegend=False,
        font=dict(family=font_type, size=size_tick_lables, color=text_col),
        margin=dict(l=100, r=40, t=120, b=100),
        height=700,
        width=1400,
    )

    # Replace subplot titles with centred ones (and safe y for PNG export)
    fig.update_layout(
        annotations=[
            dict(
                text="New Published Projects Per Year",
                x=0.225, y=1.10,
                xanchor="center",
                font=dict(size=size_title, family=font_type, color=text_col),
                showarrow=False
            ),
            dict(
                text="New Project Volume Per Year",
                x=0.775, y=1.10,
                xanchor="center",
                font=dict(size=size_title, family=font_type, color=text_col),
                showarrow=False
            ),
        ]
    )

    # ------------------------------
    # Axes styling
    # ------------------------------
    min_year = years.min()
    max_year = years.max()

    # Plot 1: Projects
    fig.update_xaxes(
        title_text="Year",
        title_font=dict(size=size_axes_lables, family=font_type, color=text_col),
        tickfont=dict(size=size_tick_lables, color=tick_col),
        showline=True, linecolor="#333", linewidth=1, ticks="outside",
        range=[min_year - 0.5, max_year + 0.5],
        row=1, col=1
    )
    fig.update_yaxes(
        title_text="New Projects",
        title_font=dict(size=size_axes_lables, family=font_type, color=text_col),
        tickfont=dict(size=size_tick_lables, color=tick_col),
        showline=True, linecolor="#333", linewidth=1, ticks="outside",
        separatethousands=True,
        range=[0, max_y_projects],
        row=1, col=1
    )

    # Plot 2: Volume
    fig.update_xaxes(
        title_text="Year",
        title_font=dict(size=size_axes_lables, family=font_type, color=text_col),
        tickfont=dict(size=size_tick_lables, color=tick_col),
        showline=True, linecolor="#333", linewidth=1, ticks="outside",
        range=[min_year - 0.5, max_year + 0.5],
        row=1, col=2
    )
    fig.update_yaxes(
        title_text="Gigabytes",
        title_font=dict(size=size_axes_lables, family=font_type, color=text_col),
        tickfont=dict(size=size_tick_lables, color=tick_col),
        showline=True, linecolor="#333", linewidth=1, ticks="outside",
        separatethousands=True,
        range=[0, max_y_volume],
        row=1, col=2
    )

    return fig


## Plot them!

In [ ]:
fig = plot_projects_volume_modern(projects_per_year)

fig.write_image("../figures/figure_2_projects.png")
fig.write_image("../figures/figure_2_projects.svg")